# YOLOの進化を動かして理解する

## 全体概要

YOLOは **画像全体を一度に見て物体の位置と種類を予測する one-stage detector** として始まりました。ただし、v1から現在までを管理する単一組織はありません。v5以降の番号は異なるチームが提案した系列を含むため、番号が大きいだけで常に同じ設計の直接後継、または全用途で優秀、という意味ではありません。

本教材は番号付きの主要リリースを v1→v13 の順に比較し、最後に Ultralytics の現行製品系列 YOLO26 を扱います。**v14〜v25という公開版があるわけではなく、YOLO26は年に合わせた命名**です。YOLOX、YOLO-NAS、RT-DETRなどの重要な派生は、番号順という今回の範囲から除外します。

### 学び方

1. 各節の詳細説明で、検出の考え方がどう変わったかを読む
2. NumPyだけの小規模デモで、その変化を最小例として観察する
3. 対応する公式実装・重みで実画像推論を行い、複数画像で結果を比較する

小規模デモは本物のネットワーク精度を再現するものではなく、差分となる演算・データ表現を分離して観察する教材です。精度や速度の数値比較には、同じデータ、解像度、ハードウェア、測定条件が必要です。

In [ ]:
from pathlib import Path
import sys, json

ROOT = Path.cwd()
if not (ROOT / 'common').exists():
    raise RuntimeError('このノートブックをリポジトリ直下から実行してください')
sys.path.insert(0, str(ROOT))

from common.versions import VERSIONS
from common.demos import run_demo
from common.runner import main as run_yolo
from common.notebook_helpers import reload_tutorial_helpers
from common.related_tech import *
from common.yolo_demo import *
from common.visualization import *

# ヘルパを編集中に変更を反映したい場合だけ実行します。
reload_tutorial_helpers(globals())

print('収録:', ', '.join(VERSIONS[v]['title'] for v in VERSIONS))


## 推論環境

技術デモのみ: `python -m pip install -r requirements-demo.txt`

画像推論も行う: `python -m pip install -r requirements.txt`

- v1〜v4: Darknet形式の `.cfg` と `.weights` を用意し、OpenCV DNNで実行
- v5/v8〜v12/YOLO26: `ultralytics` が読める対応 `.pt` を指定
- v6/v7/v13: 公式リポジトリごとに環境が異なるので、その推論コマンドを `--command` 経由で実行

この教材の実行入口は `common.runner.main` です。ノートブックでは `run_yolo(version, args)` として直接呼び出します。

## 関連技術

YOLOの各世代そのものではありませんが、物体検出を理解・実装・比較するときに必要になる周辺技術をここに整理します。たとえば評価指標、データ拡張、バックボーン、特徴ピラミッド、NMS、モデル圧縮、量子化、ONNX/TensorRT、アノテーション形式などは、このセクションに説明を追加していきます。

### Leaky ReLU

Leaky ReLUは、ReLUの負の領域を完全に0にせず、小さな傾きで通す活性化関数です。

$$
f(x) =
\begin{cases}
x & (x > 0) \\
\alpha x & (x \le 0)
\end{cases}
$$

通常のReLUでは、入力が負のとき出力も勾配も0になります。一方、Leaky ReLUでは負の側にも `alpha` 倍の小さな勾配が残ります。

YOLOv1では多くの層でLeaky ReLUを使います。負の側にも勾配が残るため、ユニットが完全に更新されなくなる問題を少し避けやすくなります。


In [ ]:
leaky_relu_fig = display_leaky_relu_demo(alpha=0.1)

### Darknet形式

Darknetは、初期YOLOの作者が使っていたC実装の深層学習フレームワークです。YOLOv1〜v4系の古いモデルでは、PyTorchの `.pt` ではなく、主に次の2つのファイルを組み合わせて推論します。

- `.cfg`: ネットワーク構造を表すテキストファイルです。層の種類、filter数、anchor、class数、入力サイズなどが入ります。
- `.weights`: 学習済み重みを入れたバイナリファイルです。畳み込みカーネル、bias、BatchNorm統計量などが入ります。

つまり、`.cfg` が「どんなネットワークか」、`.weights` が「学習済みパラメータはいくつか」を持ちます。OpenCV DNNなどでDarknet形式を読むときも、この2つを合わせてモデルを復元します。


In [ ]:
darknet_roles = darknet_file_roles()
darknet_fig = display_darknet_format_demo()

### NMS

NMS（Non-Maximum Suppression）は、同じ物体に対して重複したbbox候補が複数出たときに、スコアが高いbboxだけを残す後処理です。

基本的なアルゴリズムは次の通りです。

1. bbox候補をconfidenceやclass scoreで高い順に並べる。
2. もっともスコアが高いbboxを採用する。
3. 採用したbboxとIoUが高い他のbboxを削除する。
4. 残ったbboxに対して同じ処理を繰り返す。

YOLOv1でも、最終的な検出結果を整理するためにNMSを使います。モデル本体がbbox候補を出し、NMSが重複候補を整理する、という役割分担です。


In [ ]:
nms_result = run_nms_demo(iou_threshold=0.5)
for step in nms_result['steps']:
    print(step)
nms_fig = display_nms_demo(nms_result)

## YOLOv1 (2016)

**新しく加わった技術:** 画像全体を1回のCNNで処理し、グリッドごとにbboxとクラスを同時回帰。

ここでは、YOLOv1で重要だった3つの要素を順番に実装して、データがどう変化するかを確認します。実際のYOLOv1の重みを再現するのではなく、手元の画像と正規化bboxを使って、YOLOv1の出力表現を小さく作ります。

1. **画像全体を1回で処理:** 画像全体を入力として、`S x S x (B*5+C)` の出力テンソルを一度に作る。
2. **グリッド責任セル:** 物体中心が入ったセルを、その物体の担当セルにする。
3. **bboxとクラスの同時回帰:** 担当セルに `[x_cell, y_cell, w, h, confidence]` とclass確率を同時に埋め込む。

**注意:** この節のデモはYOLOv1の表現形式を理解するための教材です。モデル全体の再実装や学習済み重みの再現ではありません。

出典: [公式/著者実装](https://github.com/pjreddie/darknet) / [論文](https://arxiv.org/abs/1506.02640)


### YOLOv1のアーキテクチャ全体像

YOLOv1は、画像全体を1つのCNNに入れ、最後に `S x S x (B * 5 + C)` のテンソルを出力するone-stage detectorです。ネットワークは大まかに **24個の畳み込み層と2個の全結合層** からなり、多くの層でLeaky ReLUを使います。**グリッドごとに別々のCNNを走らせるのではなく、画像全体に対して1つのCNNを1回だけ実行し、その出力を `S x S` 個のセルとして解釈します。** 候補領域を先に作るtwo-stage detectorとも違い、1回のforwardで「どこに物体があるか」と「それが何か」を同時に出します。

下の図は、YOLOv1論文のFig.3のように、入力から出力までの層の並びを簡略化して描いたものです。処理の流れは次の通りです。

1. **入力画像を固定サイズにリサイズする**  
   論文では画像を `448 x 448` にリサイズしてCNNへ入れます。

2. **CNNで画像全体の特徴を抽出する**  
   畳み込み層で画像全体の特徴を作り、最後に全結合層で検出用の出力テンソルへ変換します。この時点で全グリッドセル分の予測がまとめて作られます。全結合層にはDropoutも使われます。

3. **ImageNet分類事前学習から検出fine-tuningへ移る**  
   YOLOv1は、まずImageNetで分類モデルとして事前学習し、その後に検出タスクへfine-tuningします。これは学習戦略として非常に重要です。最初から少ない検出データだけで学習するのではなく、分類で得た一般的な画像特徴を検出へ転用します。

4. **画像を `S x S` グリッドとして扱う**  
   YOLOv1では典型的に `S=7` なので、画像を `7 x 7` の49セルに分けます。物体の中心が入ったセルが、その物体の予測を担当します。

5. **各セルがbbox候補とclass確率を出す**  
   各セルは `B` 個のbbox候補を出します。各bbox候補は `x, y, w, h, confidence` を持ちます。さらに、そのセルごとにclass確率 `C` 個を出します。

6. **bbox・confidence・classを同じlossで同時に学習する**  
   bbox座標、object confidence、no-object confidence、class確率を、maskと重み付きのMSEでまとめて最適化します。

7. **推論時はconfidenceとclass確率から最終スコアを作る**  
   各bboxのconfidenceとclass確率を組み合わせ、閾値処理やNMSで最終的な検出結果を残します。

この図で重要なのは、`7 x 7` の各セルが別々のCNNで計算されるのではなく、最後の出力テンソルをセルごとに解釈している点です。この後の3つのサブセクションでは、この全体像のうち、YOLOv1の核になる「1回の出力テンソル」「グリッド責任セル」「bboxとclassの同時学習」を順番に小さく実装して確認します。


In [ ]:
v1_stages = build_yolov1_stages('images/sample.jpg')
arch_fig = display_yolov1_architecture(v1_stages)

### 1. 入力画像1枚から、出力テンソル全体を1回で得る

このサブセクションで確認したいことは、**YOLOv1は入力画像1枚に対して、画像全体の検出結果を入れるための大きな出力テンソルを1回で返す**という点です。ここでは実物のCNNは動かさず、YOLOv1が返すはずの出力の「形」と「各場所の意味」を小さく再現します。

YOLOv1の出力形状は次のように表します。

```text
S x S x (B * 5 + C)
```

それぞれの意味は次の通りです。

- `S`: 画像を縦横に何分割するか。YOLOv1では典型的に `S=7` なので、画像を `7 x 7 = 49` 個のセルに分けます。
- `B`: 1つのセルが出せるbbox候補の数。ここでは `B=2` なので、各セルはbbox候補を2個持ちます。
- `5`: bbox候補1個あたりの値の数です。内訳は `x`, `y`, `w`, `h`, `confidence` の5つです。
- `C`: クラス数です。ここではデモを小さくするため、`person`, `bicycle`, `motorcycle`, `car`, `bus` の5クラスだけを使うので `C=5` です。

したがって、このデモでは1つのセルが持つ値の数は次のようになります。

```text
B * 5 + C = 2 * 5 + 5 = 15
```

つまり出力全体は次の形です。

```text
7 x 7 x 15
```

これは「49個のセルそれぞれに、bbox候補2個分の情報と、5クラス分の確率が入る」という意味です。重要なのは、物体ごとに画像を切り出して何度も分類するのではなく、**画像全体を1回処理して、この `7 x 7 x 15` 全体をまとめて得る**という点です。


In [ ]:
v1_stages = build_yolov1_stages('images/sample.jpg')
flow_fig = display_yolov1_output_flow(v1_stages)

### 2. グリッド責任セルを決める

ここでいう「物体中心」は、**学習時にはアノテーション済みの正解bboxから計算する中心点**です。つまり、YOLOv1が最初に中心点を見つけてから担当セルを決めるわけではありません。

学習時の流れは次の通りです。

1. 正解bbox `x1, y1, x2, y2` から中心点 `(cx, cy)` を計算する。
2. その中心点が入っているgrid cellを、その物体の担当セルにする。
3. その担当セルだけに、その物体のbboxとclassを予測するように教師信号を入れる。

推論時には正解bboxはありません。その代わり、学習によって各セルは「自分のセル内に中心がある物体」を予測するようになります。各セルは `x_cell, y_cell` を出力しますが、これは画像全体での中心点ではなく、**そのセルの左上を基準にしたセル内相対座標**です。

したがって、このサブセクションで行っている処理は「中心点を予測する処理」ではなく、**正解bboxから担当セルを作り、YOLOv1がどのセルにどの物体を学習させるかを決める処理**です。


In [ ]:
for obj in v1_stages['stage2_responsible_cells']:
    print(
        f"{obj['label']:10s} center={obj['center']} "
        f"-> cell(row, col)={obj['cell']} "
        f"offset_in_cell={obj['cell_offset_xy']} "
        f"encoded={obj['encoded']}"
    )
responsible_fig = display_yolov1_responsible_cells(v1_stages)

### 3. bboxとクラスを同時に回帰する

YOLOv1の「同時に回帰する」は、**1つのセルの出力ベクトルにbbox、confidence、class確率をまとめて入れ、それらを1つの損失関数で同時に学習する**という意味です。

このデモでは、1つのセルのベクトルは次の15値です。

```text
[box1_x, box1_y, box1_w, box1_h, box1_conf,
 box2_x, box2_y, box2_w, box2_h, box2_conf,
 p_person, p_bicycle, p_motorcycle, p_car, p_bus]
```

実際のYOLOv1では、各セルは `B` 個のbbox候補を出します。ただし学習時には、その中で正解bboxと最もIoUが高いbbox predictorだけを、その物体のbbox担当にします。class確率はbboxごとではなくセルごとに予測します。

損失関数としては、CrossEntropyではなく、基本的に **MSE（Mean Squared Error / squared error）** を使います。ただし、すべての出力に同じMSEをかけるのではなく、担当するセル・bbox predictorだけをmaskし、座標誤差とno-object誤差には重みを付けます。

まず、各項を関数名ベースで書くと次のようになります。

$$
L_{xy} = MSE\left((x, y), (\hat{x}, \hat{y})\right)
$$

$$
L_{wh} = MSE\left((\sqrt{w}, \sqrt{h}), (\sqrt{\hat{w}}, \sqrt{\hat{h}})\right)
$$

$$
L_{obj} = MSE\left(C, \hat{C}\right)
$$

$$
L_{noobj} = MSE\left(0, \hat{C}\right)
$$

$$
L_{class} = MSE\left(p(class), \hat{p}(class)\right)
$$

全体としては、これらをmaskと重み付きで足し合わせます。

$$
L
= \lambda_{coord}(L_{xy} + L_{wh})
+ L_{obj}
+ \lambda_{noobj}L_{noobj}
+ L_{class}
$$

ここで、各項の意味は次の通りです。

| 項 | 何を比べるか | 損失の種類 | どこに適用するか |
|---|---|---|---|
| $L_{xy}$ | bbox中心 `x, y` | MSE | 物体を担当するbbox predictorだけ |
| $L_{wh}$ | bboxサイズ `sqrt(w), sqrt(h)` | MSE | 物体を担当するbbox predictorだけ |
| $L_{obj}$ | object confidence | MSE | 物体を担当するbbox predictorだけ |
| $L_{noobj}$ | no-object confidence | MSE | 物体を担当しないbbox predictor |
| $L_{class}$ | class確率 | MSE | 物体中心を含む担当セルだけ |

ここでいう「mask」とは、**その損失を計算する場所だけ1、それ以外を0にするスイッチ**です。YOLOv1の論文では、このスイッチを indicator function として書きます。たとえば、`obj mask` は「そのbbox predictorが物体を担当するなら有効、そうでなければ無効」という意味です。

重みは主に次の2つです。

- `lambda_coord = 5`: bbox座標の誤差を強める。位置合わせを重視するためです。
- `lambda_noobj = 0.5`: 物体がない場所のconfidence誤差を弱める。画像内の大半のセルは物体なしなので、その損失が支配的になりすぎるのを防ぎます。

bboxサイズだけ `w, h` をそのままMSEにせず、`sqrt(w), sqrt(h)` にしてからMSEを取ります。これは、大きい箱の少しのずれより、小さい箱の少しのずれを相対的に重く見るためです。

上の表で使っている値の意味は次の通りです。

- `x, y`: 担当セル内でのbbox中心座標。セル左上を基準にした0〜1の相対座標です。
- `w, h`: 画像全体に対するbbox幅・高さです。
- `confidence`: そのbbox predictorが物体を含む確からしさです。学習時には、担当bbox predictorではIoUに近い値を目標にし、物体がない場所では0へ近づけます。
- `p(class)`: class確率です。担当セルだけがclass lossを持ちます。

つまり、特別な後処理でbboxとclassを別々に作っているのではありません。**同じ出力テンソルの中にbbox回帰値とclass確率が同居しており、損失関数の各項がそれぞれの場所を同時に更新する**、というアルゴリズムです。


In [ ]:
print(json.dumps(v1_stages['stage3_joint_regression'], ensure_ascii=False, indent=2))
loss_demo = compute_yolov1_loss_demo(v1_stages)
print(json.dumps({
    'loss_weights': {
        'lambda_coord': loss_demo['lambda_coord'],
        'lambda_noobj': loss_demo['lambda_noobj'],
    },
    'first_object_loss_example': loss_demo['rows'][0]['weighted_loss'],
    'noobject_confidence_example': loss_demo['noobject_example'],
    'total_loss_breakdown': loss_demo['totals'],
}, ensure_ascii=False, indent=2))
stage3_fig = display_yolov1_stages(v1_stages)

## YOLOv2 / YOLO9000 (2017)

**新しく加わった技術:** anchor box、k-meansによるbox事前分布、BN、passthrough、階層分類。

**詳細:**

YOLOv2は、YOLOv1の直接bbox回帰をanchor boxベースの予測へ寄せました。anchorは「よく出る箱の形」を事前に用意し、モデルはその箱からのずれを予測します。これにより、縦長・横長・大きい・小さいといった物体形状を最初から複数仮定でき、位置合わせが安定します。さらにBatch Normalization、高解像度分類器の事前学習、passthrough層による細かい特徴の利用などで精度を底上げしました。

デモでは、物体の幅高さとanchor候補の近さを面積比で比べ、どのanchorが担当しやすいかを見ます。ここで見ているのは、学習前からモデルに与えられる「箱の形の事前知識」です。YOLOv1のセル中心だけの担当から、セルとanchorの組み合わせによる担当へ移ったことが重要です。

**その結果使いやすくなった場面:** 形状の異なる物体と多数カテゴリ（検出・分類データの共同学習）。

**注意:** この節のデモは `anchors` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/pjreddie/darknet) / [論文](https://arxiv.org/abs/1612.08242)

In [ ]:
# YOLOv2 / YOLO9000 の技術差分デモ
print(json.dumps(run_demo(2), ensure_ascii=False, indent=2))

### 実際の推論体験

重み名・公式CLIの引数は、リンク先で取得した版に合わせてください。ノートブック内では `common.runner.main` を直接呼びます。

```python
run_yolo(2, ['--mode', 'infer', '--image', 'images/sample.jpg', '--config', 'path/to/yolov2.cfg', '--weights', 'path/to/yolov2.weights'])
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLOv3 (2018)

**新しく加わった技術:** Darknet-53、3スケール予測、FPN的な特徴融合、独立ロジスティック分類。

**詳細:**

YOLOv3は、1つの粗い特徴マップだけでなく、複数スケールの特徴マップで検出する方向へ進みました。深い層は意味情報に強い一方で解像度が低く、小物体の位置が粗くなります。浅い層は細かい位置情報を残しますが、意味理解は弱くなります。YOLOv3はこの性質を利用し、stride 8、16、32のような異なる解像度で予測します。

デモでは、入力416に対して52x52、26x26、13x13の検出グリッドができることを確認します。小さい物体は細かい52x52、大きい物体は粗い13x13で扱いやすくなります。ここから、現代的な検出器で当たり前になったmulti-scale predictionの意味が見えてきます。

**その結果使いやすくなった場面:** 小物体を含む監視映像や交通映像など、サイズ差の大きい対象。

**注意:** この節のデモは `multiscale` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/pjreddie/darknet) / [論文](https://arxiv.org/abs/1804.02767)

In [ ]:
# YOLOv3 の技術差分デモ
print(json.dumps(run_demo(3), ensure_ascii=False, indent=2))

### 実際の推論体験

重み名・公式CLIの引数は、リンク先で取得した版に合わせてください。ノートブック内では `common.runner.main` を直接呼びます。

```python
run_yolo(3, ['--mode', 'infer', '--image', 'images/sample.jpg', '--config', 'path/to/yolov3.cfg', '--weights', 'path/to/yolov3.weights'])
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLOv4 (2020)

**新しく加わった技術:** CSPDarknet53、SPP+PAN、Mish、CIoU、Mosaic等のBoF/BoSを統合。

**詳細:**

YOLOv4は、モデル構造だけでなく学習時の工夫を体系的に組み合わせた世代です。CSPDarknet53、SPP、PANのような特徴抽出・特徴融合に加え、Mosaic、CIoU loss、DropBlock、SATなど、精度を上げるためのBag of Freebies/Bag of Specialsを積極的に統合しました。単に深いネットワークにするのではなく、GPUで現実的に学習しやすく、かつ推論を速く保つ設計が重視されています。

デモではMosaic augmentationの考え方を、4枚の小さな画像を1枚に組み合わせる操作として観察します。Mosaicは1回の入力に複数の文脈とスケールを混ぜるため、小物体や背景変化に強くなりやすい一方、学習分布を人工的に広げる操作でもあります。

**その結果使いやすくなった場面:** 単一GPUで学習しつつ高精度な産業・交通向けリアルタイム検出。

**注意:** この節のデモは `mosaic` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/AlexeyAB/darknet) / [論文](https://arxiv.org/abs/2004.10934)

In [ ]:
# YOLOv4 の技術差分デモ
print(json.dumps(run_demo(4), ensure_ascii=False, indent=2))

### 実際の推論体験

重み名・公式CLIの引数は、リンク先で取得した版に合わせてください。ノートブック内では `common.runner.main` を直接呼びます。

```python
run_yolo(4, ['--mode', 'infer', '--image', 'images/sample.jpg', '--config', 'path/to/yolov4.cfg', '--weights', 'path/to/yolov4.weights'])
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLOv5 (2020)

**新しく加わった技術:** PyTorch実装、CSP系backbone、PAN-FPN、AutoAnchor、使いやすい学習・配備API。

**詳細:**

YOLOv5は論文中心の単一技術というより、PyTorch実装としての扱いやすさ、学習・評価・export・配備までの一貫した開発体験を広めた点が大きい世代です。CSP系backbone、PAN-FPN、AutoAnchor、豊富なaugmentationなどを実務で使いやすい形にまとめ、独自データで素早く検出器を作る流れを一般化しました。

デモではFocus層の発想を見ます。画像の偶数行・奇数行・偶数列・奇数列をチャンネル方向へ並べ替えることで、空間解像度を下げながら情報をチャンネルに移します。これは初期のYOLOv5で使われた効率化の例で、空間サイズ、チャンネル数、計算量のトレードオフを考える材料になります。

**その結果使いやすくなった場面:** 独自データの迅速な学習、ONNX/TensorRT等への実務配備。

**注意:** この節のデモは `focus` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/ultralytics/yolov5)

In [ ]:
# YOLOv5 の技術差分デモ
print(json.dumps(run_demo(5), ensure_ascii=False, indent=2))

### 実際の推論体験

重み名・公式CLIの引数は、リンク先で取得した版に合わせてください。ノートブック内では `common.runner.main` を直接呼びます。

```python
run_yolo(5, ['--mode', 'infer', '--image', 'images/sample.jpg', '--weights', 'yolov5nu.pt', '--save'])
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLOv6 (2022)

**新しく加わった技術:** RepVGG系の再パラメータ化、効率的decoupled head、SimOTA/TAL系割当。

**詳細:**

YOLOv6は、産業用途での低遅延推論を強く意識し、学習時と推論時の構造を分ける再パラメータ化を取り入れました。学習中は3x3や1x1など複数枝を使って表現力と最適化しやすさを確保し、推論時にはそれらを1つの畳み込みに畳み込んで高速化します。効率的なdecoupled headやラベル割当の改善も、速度と精度の両立に効きます。

デモでは、3x3カーネルと1x1カーネルを推論用の1つの3x3カーネルへ融合する様子を見ます。ポイントは、学習しやすい形と推論しやすい形が必ずしも同じでなくてよい、という設計思想です。

**その結果使いやすくなった場面:** 物流・製造などGPU推論の低遅延が要求される産業用途。

**注意:** この節のデモは `reparam` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/meituan/YOLOv6) / [論文](https://arxiv.org/abs/2209.02976)

In [ ]:
# YOLOv6 の技術差分デモ
print(json.dumps(run_demo(6), ensure_ascii=False, indent=2))

### 実際の推論体験

重み名・公式CLIの引数は、リンク先で取得した版に合わせてください。ノートブック内では `common.runner.main` を直接呼びます。

```python
run_yolo(6, ['--mode', 'infer', '--image', 'images/sample.jpg', '--weights', 'path/to/model', '--command', 'python path/to/official/infer.py --weights {weights} --source {image}'])
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLOv7 (2022)

**新しく加わった技術:** E-ELAN、モデルscaling、予定再パラメータ化、trainable bag-of-freebies。

**詳細:**

YOLOv7は、E-ELANによって勾配の流れと特徴の多様性を保ちながらネットワークを拡張する設計を示しました。複数経路で特徴を変換し、それらを結合することで表現を豊かにしつつ、学習が破綻しにくい構造を狙います。また、planned re-parameterizationやモデルスケーリングの整理により、サイズ違いのモデルを作るときの一貫性も重視されました。

デモでは、同じ入力から複数の経路を作り、最後にconcatする様子を観察します。これはELAN系の「特徴変換の経路を増やしてから統合する」直感を小さくしたものです。検出器の改良がheadだけでなく、backbone内部の情報の流し方にも及ぶことがわかります。

**その結果使いやすくなった場面:** エッジからGPUサーバまで、速度と精度の異なる制約へスケール。

**注意:** この節のデモは `elan` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/WongKinYiu/yolov7) / [論文](https://arxiv.org/abs/2207.02696)

In [ ]:
# YOLOv7 の技術差分デモ
print(json.dumps(run_demo(7), ensure_ascii=False, indent=2))

### 実際の推論体験

重み名・公式CLIの引数は、リンク先で取得した版に合わせてください。ノートブック内では `common.runner.main` を直接呼びます。

```python
run_yolo(7, ['--mode', 'infer', '--image', 'images/sample.jpg', '--weights', 'path/to/model', '--command', 'python path/to/official/infer.py --weights {weights} --source {image}'])
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLOv8 (2023)

**新しく加わった技術:** anchor-free decoupled head、C2f、タスク統合API（検出・分割・姿勢等）。

**詳細:**

YOLOv8はanchor-freeな検出ヘッドを採用し、anchor候補の形に縛られず、特徴マップ上の点からbboxまでの距離を直接予測する形へ移りました。anchor設計やAutoAnchorのような前処理への依存が減り、検出・セグメンテーション・姿勢推定などを同じAPIで扱える実装体系も整いました。C2fなどの構造も、特徴再利用と効率のバランスを狙っています。

デモでは、anchor pointから左・上・右・下への距離を使ってbboxを復元します。YOLOv2以降の「anchorの幅高さからずれを予測する」考え方と比べると、箱の形の事前候補ではなく、点と距離で箱を表す発想に変わったことが見どころです。

**その結果使いやすくなった場面:** 同じ操作体系で検出、segmentation、pose、trackingを試す製品開発。

**注意:** この節のデモは `anchor_free` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/ultralytics/ultralytics)

In [ ]:
# YOLOv8 の技術差分デモ
print(json.dumps(run_demo(8), ensure_ascii=False, indent=2))

### 実際の推論体験

重み名・公式CLIの引数は、リンク先で取得した版に合わせてください。ノートブック内では `common.runner.main` を直接呼びます。

```python
run_yolo(8, ['--mode', 'infer', '--image', 'images/sample.jpg', '--weights', 'yolov8n.pt', '--save'])
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLOv9 (2024)

**新しく加わった技術:** Programmable Gradient Information (PGI) と GELAN。

**詳細:**

YOLOv9は、深いネットワークで情報が失われたり勾配が弱くなったりする問題に対して、Programmable Gradient Information (PGI) という補助的な学習経路を導入しました。推論時に必要な本体経路を重くしすぎず、学習時にはより豊かな勾配情報を流すことで、軽量モデルでも学習を助けます。GELANはELAN系の発想を発展させた効率的な特徴抽出構造です。

デモでは、主経路の勾配に補助枝からの情報が足される様子を数値で見ます。重要なのは、補助枝は推論時の計算を増やすためのものではなく、学習時に本体をよく育てるための仕掛けだという点です。

**その結果使いやすくなった場面:** 軽量モデルでも学習時の情報損失を抑えたいエッジ検出。

**注意:** この節のデモは `pgi` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/WongKinYiu/yolov9) / [論文](https://arxiv.org/abs/2402.13616)

In [ ]:
# YOLOv9 の技術差分デモ
print(json.dumps(run_demo(9), ensure_ascii=False, indent=2))

### 実際の推論体験

重み名・公式CLIの引数は、リンク先で取得した版に合わせてください。ノートブック内では `common.runner.main` を直接呼びます。

```python
run_yolo(9, ['--mode', 'infer', '--image', 'images/sample.jpg', '--weights', 'yolov9c.pt', '--save'])
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLOv10 (2024)

**新しく加わった技術:** consistent dual assignmentsによるend-to-end NMS-free検出と効率設計。

**詳細:**

YOLOv10は、NMSに頼らないend-to-end検出を強く打ち出しました。従来は多くの重複bboxを出してからNMSで消す流れが一般的でしたが、NMSは後処理として遅延や実装差を生みます。YOLOv10はone-to-manyとone-to-oneの割当を組み合わせ、一方で豊富な学習信号を確保しながら、推論では重複の少ない直接的な予測を目指します。

デモでは、重複した箱をNMSで消す例と、one-to-oneの訓練ターゲットが少数の直接予測へ向かう例を対比します。ここで学ぶべき変化は、検出性能だけでなく、後処理を含めたシステム全体の遅延を設計対象にしたことです。

**その結果使いやすくなった場面:** NMS遅延を避けたいストリーミング・組込みリアルタイム推論。

**注意:** この節のデモは `nms_free` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/THU-MIG/yolov10) / [論文](https://arxiv.org/abs/2405.14458)

In [ ]:
# YOLOv10 の技術差分デモ
print(json.dumps(run_demo(10), ensure_ascii=False, indent=2))

### 実際の推論体験

重み名・公式CLIの引数は、リンク先で取得した版に合わせてください。ノートブック内では `common.runner.main` を直接呼びます。

```python
run_yolo(10, ['--mode', 'infer', '--image', 'images/sample.jpg', '--weights', 'yolov10n.pt', '--save'])
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLO11 (2024)

**新しく加わった技術:** C3k2/C2PSA等で特徴抽出とattentionを効率化しマルチタスク性能を改善。

**詳細:**

YOLO11は、Ultralytics系列の実装として、C3k2やC2PSAなどのモジュールで特徴抽出とattentionを効率化し、検出以外のタスクも含めた実用性を高めています。CNNの局所的な特徴抽出だけでなく、重要な位置同士の関係を軽量に取り込む方向がより前面に出ています。

デモでは、少数のtoken間でattention重みを計算し、各行の重みが1に正規化されることを確認します。これは実モデルそのものではありませんが、ある位置の特徴が他の位置を参照して更新される、というattentionの基本動作を検出器の文脈で見るための最小例です。

**その結果使いやすくなった場面:** 限られた計算資源で検出・分割・姿勢・OBB・分類を横断利用。

**注意:** この節のデモは `attention` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/ultralytics/ultralytics)

In [ ]:
# YOLO11 の技術差分デモ
print(json.dumps(run_demo(11), ensure_ascii=False, indent=2))

### 実際の推論体験

重み名・公式CLIの引数は、リンク先で取得した版に合わせてください。ノートブック内では `common.runner.main` を直接呼びます。

```python
run_yolo(11, ['--mode', 'infer', '--image', 'images/sample.jpg', '--weights', 'yolo11n.pt', '--save'])
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLOv12 (2025)

**新しく加わった技術:** Area Attention (A2) と R-ELANでattentionをリアルタイム検出向けに最適化。

**詳細:**

YOLOv12は、attentionをリアルタイム検出へ入れるときの計算量を抑えるため、Area Attentionのように空間を領域へ分けて処理する設計を取り入れます。全token同士でattentionを取ると計算量が大きくなりますが、領域内に制限すれば文脈利用と速度の折り合いをつけやすくなります。R-ELANも効率的な特徴伝播を支えます。

デモでは、8個のtokenで全体attentionを行う場合の組み合わせ数と、2領域に分けた場合の組み合わせ数を比べます。小さな数値例でも、globalな関係をすべて見ることと、局所領域に分けることの計算量差がはっきり出ます。

**その結果使いやすくなった場面:** 混雑場面など広い文脈が必要で、CNN並みの低遅延も必要な検出。

**注意:** この節のデモは `area_attention` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/sunsmarterjie/yolov12) / [論文](https://arxiv.org/abs/2502.12524)

In [ ]:
# YOLOv12 の技術差分デモ
print(json.dumps(run_demo(12), ensure_ascii=False, indent=2))

### 実際の推論体験

重み名・公式CLIの引数は、リンク先で取得した版に合わせてください。ノートブック内では `common.runner.main` を直接呼びます。

```python
run_yolo(12, ['--mode', 'infer', '--image', 'images/sample.jpg', '--weights', 'yolo12n.pt', '--save'])
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## YOLOv13 (2025)

**新しく加わった技術:** HyperACEによるhypergraph特徴相関、FullPAD、DS-based label assignment。

**詳細:**

YOLOv13は、特徴間の関係を単純なペアではなく、hypergraphとして扱う方向を示します。通常のgraphが2点間の辺を扱うのに対し、hypergraphは複数のノードをまとめるhyperedgeを持てます。遮蔽された物体、密集した物体、離れた部位の関係など、複数特徴のまとまりを扱いたい場面で有効になる可能性があります。

デモでは、ノード特徴をhyperedgeへ集約し、さらにノードへ戻す伝播を行います。これはHyperACEの直感を最小化したもので、畳み込みやattentionとは別の「高次の特徴相関をまとめて戻す」考え方を確認するための例です。

**その結果使いやすくなった場面:** 遮蔽・密集・離れた部位の関係など高次の特徴相関が重要な場面。

**注意:** この節のデモは `hypergraph` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/iMoonLab/yolov13) / [論文](https://arxiv.org/abs/2506.17733)

In [ ]:
# YOLOv13 の技術差分デモ
print(json.dumps(run_demo(13), ensure_ascii=False, indent=2))

### 実際の推論体験

重み名・公式CLIの引数は、リンク先で取得した版に合わせてください。ノートブック内では `common.runner.main` を直接呼びます。

```python
run_yolo(13, ['--mode', 'infer', '--image', 'images/sample.jpg', '--weights', 'path/to/model', '--command', 'python path/to/official/infer.py --weights {weights} --source {image}'])
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## 補足: YOLO26 (Ultralytics product line) (2026)

**新しく加わった技術:** end-to-end NMS-free推論、DFL除去、ProgLoss+STAL、MuSGDによる学習最適化。

**詳細:**

YOLO26は、番号順のYOLOv14という意味ではなく、Ultralyticsの製品系列として配備効率とend-to-end性を前面に出したものとして扱います。NMS-free推論、DFL除去、学習最適化などにより、モデル本体だけでなくCPU・エッジ・export後の扱いやすさまで含めた効率を重視します。

デモでは、推論後処理がNMSではなくconfidence threshold中心に単純化される様子を見ます。検出器の進化が、精度指標だけでなく、配備時の遅延、後処理の複雑さ、実装の一貫性へ広がっていることを確認する節です。

**その結果使いやすくなった場面:** CPU・エッジ機器への配備と、後処理を単純化した低遅延end-to-end推論。

**注意:** この節のデモは `deployment` の要点だけを可視化します。モデル全体の再実装ではありません。

出典: [公式/著者実装](https://github.com/ultralytics/ultralytics) / [公式ドキュメント](https://docs.ultralytics.com/models/yolo26/)

In [ ]:
# YOLO26 (Ultralytics product line) の技術差分デモ
print(json.dumps(run_demo(26), ensure_ascii=False, indent=2))

### 実際の推論体験

重み名・公式CLIの引数は、リンク先で取得した版に合わせてください。ノートブック内では `common.runner.main` を直接呼びます。

```python
run_yolo(26, ['--mode', 'infer', '--image', 'images/sample.jpg', '--weights', 'yolo26n.pt', '--save'])
```

`--mode info` は世代情報、`--mode demo` は上と同じ小規模デモを表示します。外部実装用の `--command` はシェルを介さず引数列として実行されます。信頼できるコマンドだけを指定してください。

## 複数画像で結果を比較する

以下では選んだ世代を複数画像に対して実行します。`VERSION`, `IMAGE_PATHS`, `WEIGHTS` を変更してください。v1〜v4では `CONFIG` も必要です。推論後は `runs/detect/predict*` に保存された結果画像を横並びで表示します。

In [ ]:
VERSION = 8
IMAGE_PATHS = [
    'images/sample.jpg',
    'images/pizza_table.jpg',
    'images/street_dog.jpg',
    'images/soccer_kids.jpg',
    'images/kitchen_cat.jpg',
]
IMAGE_SOURCE = 'images'
WEIGHTS = 'yolov8n.pt'
CONFIG = None

args = ['--mode', 'infer', '--image', IMAGE_SOURCE, '--weights', WEIGHTS, '--save']
if CONFIG:
    args += ['--config', CONFIG]
print('runner.main', VERSION, args)
# ダウンロードと推論を実行するときに次行のコメントを外す
# run_yolo(VERSION, args)

# 推論を実行した後に、保存された結果画像を横並びで確認します
# result_images = prediction_images_for(IMAGE_PATHS)
# display_image_grid(result_images, titles=[Path(p).name for p in IMAGE_PATHS], columns=3)


## データセット

**推論体験だけならダウンロード不要**です。`images/sample.jpg` に任意の写真を置いてください。

独自学習まで進む場合の小規模候補:

- [COCO128](https://docs.ultralytics.com/datasets/detect/coco128/): COCO train2017の先頭128枚。パイプライン確認向けで、性能評価には小さすぎます。
- [Pascal VOC](http://host.robots.ox.ac.uk/pascal/VOC/): v1/v2時代との比較に適した20クラス。
- [COCO](https://cocodataset.org/#download): 標準的な80クラス。容量・学習時間が大きい本評価向け。

データ利用条件と画像の権利を確認してください。世代比較では同じtrain/validation分割、入力解像度、augmentation、評価指標（通常 COCO mAP）を固定します。

## まとめ

- v1〜v3: 統一回帰 → anchor → multi-scale と、検出器の基本形を確立
- v4〜v7: 学習手法、特徴融合、再パラメータ化で速度精度比を改善
- v8〜v10: anchor-free、勾配設計、NMS-free end-to-endへ移行
- v11〜v13: attentionと高次特徴相関をリアルタイム制約へ導入
- YOLO26: 番号連番ではない製品系列として、配備とend-to-end効率を重視

「最新版」を選ぶ前に、ライセンス、対象タスク、対応export形式、実測レイテンシ、保守性を同じ重要度で確認してください。